# Ghost population model

Formålet med denne notebook er at modellere effekten af en "ghost population", dvs. en population som ikke er observeret i data, men som påvirker genetiske mønstre via migration eller admixture.

Dette er centralt for:
- at opdage skjulte populationer
- at modellere komplekse populationshistorier

## Hvad er en ghost population?

En ghost population er:
- ikke direkte observeret
- men påvirker observerede populationer via gene flow

Eksempel:
    pop1 ←→ ghost ←→ pop2

Selvom ghost ikke samples, påvirker den:
- coalescent tider
- SFS
- genetisk variation

Dette kan give:
- "uforklarlige" mønstre i data

## Modelstruktur

Jeg udvider two-island modellen til tre populationer:

- pop1 (observeret)
- pop2 (observeret)
- ghost (ikke observeret)

Transitions:
1. Coalescence inden for populationer
2. Migration mellem populationer
3. Ghost population påvirker indirekte

Observation:
Jeg observerer kun lineager fra pop1 og pop2
→ ghost er skjult

In [ ]:
nr_samples = 2

indexer = StateIndexer(
    descendants=[
        Property('pop1', max_value=nr_samples),
        Property('pop2', max_value=nr_samples),
        Property('ghost', max_value=nr_samples),
        Property('location', min_value=1, max_value=3)
    ]
)

In [ ]:
initial = [0] * indexer.state_length

# start: én lineage i hver observeret population
initial[indexer.props_to_index(pop1=1, pop2=0, ghost=0, location=1)] = 1
initial[indexer.props_to_index(pop1=0, pop2=1, ghost=0, location=2)] = 1

In [ ]:
@with_ipv(initial)
def ghost_model(state, indexer=None):
    transitions = []
    
    if state.sum() <= 1:
        return transitions
    
    for i in range(indexer.state_length):
        if state[i] == 0:
            continue
        
        pi = indexer.index_to_props(i)
        
        for j in range(i, indexer.state_length):
            if state[j] == 0:
                continue
            
            pj = indexer.index_to_props(j)
            
            same = int(i == j)
            
            # --- COALESCENCE ---
            if pi.location == pj.location:
                
                if same and state[i] < 2:
                    continue
                if not same and (state[i] < 1 or state[j] < 1):
                    continue
                
                new = state.copy()
                new[i] -= 1
                new[j] -= 1
                
                new_pop1 = pi.pop1 + pj.pop1
                new_pop2 = pi.pop2 + pj.pop2
                new_ghost = pi.ghost + pj.ghost
                
                new[new_index := indexer.props_to_index(
                    pop1=new_pop1,
                    pop2=new_pop2,
                    ghost=new_ghost,
                    location=pi.location
                )] += 1
                
                rate = state[i]*(state[j]-same)/(1+same)
                
                transitions.append([new, [rate, 0, 0, 0, 0]])
            
            # --- MIGRATION ---
            
            # pop1 → ghost
            if pi.location == 1:
                new = state.copy()
                new[i] -= 1
                
                new[indexer.props_to_index(
                    pop1=pi.pop1,
                    pop2=pi.pop2,
                    ghost=pi.ghost,
                    location=3
                )] += 1
                
                transitions.append([new, [0, 1, 0, 0, 0]])
            
            # ghost → pop1
            if pi.location == 3:
                new = state.copy()
                new[i] -= 1
                
                new[indexer.props_to_index(
                    pop1=pi.pop1,
                    pop2=pi.pop2,
                    ghost=pi.ghost,
                    location=1
                )] += 1
                
                transitions.append([new, [0, 0, 1, 0, 0]])
            
            # pop2 → ghost
            if pi.location == 2:
                new = state.copy()
                new[i] -= 1
                
                new[indexer.props_to_index(
                    pop1=pi.pop1,
                    pop2=pi.pop2,
                    ghost=pi.ghost,
                    location=3
                )] += 1
                
                transitions.append([new, [0, 0, 0, 1, 0]])
            
            # ghost → pop2
            if pi.location == 3:
                new = state.copy()
                new[i] -= 1
                
                new[indexer.props_to_index(
                    pop1=pi.pop1,
                    pop2=pi.pop2,
                    ghost=pi.ghost,
                    location=2
                )] += 1
                
                transitions.append([new, [0, 0, 0, 0, 1]])
    
    return transitions

In [ ]:
graph = Graph(ghost_model, indexer=indexer)
graph.plot()

In [ ]:
theta = 1.0
m1g = 0.5
mg1 = 0.5
m2g = 0.5
mg2 = 0.5

graph.update_weights([theta, m1g, mg1, m2g, mg2])

In [ ]:
samples = graph.sample(2000)

plt.hist(samples, bins=50, density=True)
plt.title("Ghost model coalescent times")
plt.show()

In [ ]:
ghost_strength = [0.1, 0.5, 1.0]

for g in ghost_strength:
    graph.update_weights([theta, g, g, g, g])
    samples = graph.sample(2000)
    
    plt.hist(samples, bins=40, density=True, alpha=0.4, label=f"ghost={g}")

plt.legend()
plt.title("Effect of ghost population")
plt.show()

Ghost populations kan:

- ændre coalescent tider
- skabe signaler der ligner admixture
- påvirke SFS

Selvom populationen ikke observeres direkte, kan dens effekt detekteres indirekte.